# Triton on Google Colab: Quick Start

This notebook installs **Triton**, verifies GPU/CUDA, then runs a tiny Triton kernel and a small matmul benchmark.

Steps:
1. Runtime ▶️ Change runtime type ▶️ **GPU**
2. Run cells top-to-bottom


In [12]:
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Capability:', torch.cuda.get_device_capability(0))
else:
    print('⚠️ Enable GPU: Runtime → Change runtime type → GPU')

Python: 3.12.12
PyTorch: 2.9.0+cu126
CUDA available: True
Device: Tesla T4
Capability: (7, 5)


In [13]:
# Install Triton. Pin if you want a specific version, e.g., triton==3.0.0
!pip -q install triton
import triton, triton.language as tl
print('Triton version:', triton.__version__)

Triton version: 3.5.0


In [14]:
from einops import einsum, rearrange

def _attention_and_lse(q, k, v, is_causal=False):
    n_queries = q.shape[-2]
    n_keys = k.shape[-2]
    d = q.shape[-1]
    scale = 1 / (d**0.5)
    S = einsum(q, k, "... q d, ... k d -> ... q k") * scale
    if is_causal:
        S = torch.where(
            torch.arange(n_queries, device=S.device)[None, :, None]
            >= torch.arange(n_keys, device=S.device)[None, None, :],
            S,
            -1e6,
        )
    P = torch.softmax(S, dim=-1)
    o = einsum(P, v, "... q k, ... k d -> ... q d")
    L = torch.logsumexp(S, dim=-1)
    return o, L

In [15]:
def _make_attn_inputs(device=None):
    torch.random.manual_seed(0)
    batch_size = 4
    n_queries = 128
    n_keys = 128
    D = 64
    q = torch.randn(batch_size, n_queries, D, device=device, requires_grad=True)
    k = torch.randn(batch_size, n_keys, D, device=device, requires_grad=True)
    v = torch.randn(batch_size, n_keys, D, device=device, requires_grad=True)
    do = torch.randn(batch_size, n_queries, D, device=device)

    return q, k, v, do


In [16]:
def _test_flash_forward_pass(impl, device="cpu", is_causal=False):
    q, k, v, _do = _make_attn_inputs(device)
    o = impl(q, k, v, is_causal)
    # Extract L from the saved tensors
    assert (
        o.grad_fn.saved_tensors is not None
    ), "No saved tensors found in the output tensor. Make sure your autograd forward is saving them using ctx.save_for_backward."
    maybe_ls = [
        t for t in o.grad_fn.saved_tensors if t.shape == (q.shape[0], q.shape[1])
    ]

    assert (
        len(maybe_ls) == 1
    ), f"Expected one tensor of shape {q.shape[0], q.shape[1]} in saved tensors, but found {len(maybe_ls)}. The tests require you to save exactly one tensor of this shape, corresponding to the log-sum-exp of the attention scores."
    l = maybe_ls[0]

    o_ref, l_ref = _attention_and_lse(q, k, v, is_causal)

    torch.testing.assert_close(o, o_ref, rtol=1e-2, atol=1e-2)
    torch.testing.assert_close(l, l_ref, rtol=1e-2, atol=1e-2)

In [17]:
def test_flash_forward_pass_triton(is_causal):
    _test_flash_forward_pass(
        flashAttentionFwd.apply,
        device="cuda",
        is_causal=is_causal,
    )

In [23]:
from threading import BoundedSemaphore
#FLASH ATTENTION IN TRITON
import triton
import triton.language as tl

@triton.jit
def flash_attention_fwd(
    Q_ptr,
    K_ptr,
    V_ptr,
    O_ptr,
    L_ptr,
    mask_ptr,
    stride_qb,
    stride_qq,
    stride_qd,
    stride_kb,
    stride_kk,
    stride_kd,
    stride_vb,
    stride_vk,
    stride_vd,
    stride_ob,
    stride_oq,
    stride_od,
    stride_lb,
    stride_lq,
    N_QUERIES,
    N_KEYS,
    scale,
    D: tl.constexpr,
    Q_TILE_SIZE: tl.constexpr,
    K_TILE_SIZE: tl.constexpr,
    is_causal,
):
    # This is already partitioned into query
    query_tile_index = tl.program_id(0)
    # This is the batch index. The launch grid will be (Tq,batch_size).
    # so program instance is (0,0) will run the tile 0 of Q for batch 0.
    batch_index = tl.program_id(1)

    #############
    ## Block pointer for output O,L
    #############

    # output is stored
    O_block_ptr = tl.make_block_ptr(
        O_ptr + batch_index * stride_ob,
        shape=(N_QUERIES, D),
        strides=(stride_oq, stride_od),
        offsets=(query_tile_index * Q_TILE_SIZE, 0),
        block_shape=(Q_TILE_SIZE, D),
        order=(1, 0),
    )

    # L block pointer
    L_block_ptr = tl.make_block_ptr(
        L_ptr + batch_index * stride_lb,
        shape=(N_QUERIES,),
        strides=(stride_lq,),
        offsets=(query_tile_index * Q_TILE_SIZE,),
        block_shape=(Q_TILE_SIZE,),
        order=(0,),
    )

    #############
    ## Block pointer for inputs
    #############

    # Block pointer of Q
    Q_block_ptr = tl.make_block_ptr(
        Q_ptr + batch_index * stride_qb,
        shape=(N_QUERIES, D),
        strides=(stride_qq, stride_qd),
        offsets=(query_tile_index * Q_TILE_SIZE, 0),
        block_shape=(Q_TILE_SIZE, D),
        order=(1, 0),
    )

    # Block pointer of K
    K_block_ptr = tl.make_block_ptr(
          K_ptr + batch_index * stride_kb,
          shape=(N_KEYS, D),
          strides=(stride_kk, stride_kd),
          offsets=(0, 0),
          block_shape=(K_TILE_SIZE, D),
          order=(1, 0),
      )

    # Block pointer of V
    V_block_ptr = tl.make_block_ptr(
          V_ptr + batch_index * stride_vb,
          shape=(N_KEYS, D),
          strides=(stride_vk, stride_vd),
          offsets=(0, 0),
          block_shape=(K_TILE_SIZE, D),
          order=(1, 0),
      )

    # block pointer of Mask
    mask_block_ptr = tl.make_block_ptr(
          mask_ptr,
          shape=(N_QUERIES, N_KEYS),
          strides=(N_KEYS, 1),
          offsets=(query_tile_index * Q_TILE_SIZE, 0),
          block_shape=(Q_TILE_SIZE, K_TILE_SIZE),
          order=(1, 0),
      )

    #############
    ## Block pointer for intermediate values
    #############

    # m is for this query batch
    m = tl.full(
        shape=(Q_TILE_SIZE,),
        value=-1 * float("Inf"),
        dtype=tl.float32,
    )

    # l for this query batch
    l = tl.zeros(
        shape=((Q_TILE_SIZE,)),
        dtype=tl.float32,
    )

    #############
    ## Loading values from pointer
    #############

    # load the O. Load it outside
    current_O = tl.load(O_block_ptr, boundary_check=(0, 1))
    # load Q tile. loading outside because its the same
    q_tile = tl.load(Q_block_ptr, boundary_check=(0, 1))

    # Loop over Tk
    for j in range(tl.cdiv(N_KEYS, K_TILE_SIZE)):
        # load the V
        v_tile = tl.load(V_block_ptr, boundary_check=(0, 1))
        # load K tile
        k_tile = tl.load(K_block_ptr, boundary_check=(0, 1))
        # load mask
        mask_tile = tl.load(mask_block_ptr, boundary_check=(0, 1))
        # compute score
        dot_prod = tl.dot(q_tile, tl.trans(input=k_tile))
        # normalize score
        score = dot_prod*(scale)
        if is_causal:
          score = score + mask_tile
        # Online softmax
        prev_m = m
        m = tl.maximum(prev_m, tl.max(score, axis=-1))
        # compute P
        P = tl.exp(score - m[:,None])
        # compute l
        prev_l = l
        l = tl.exp(prev_m - m) * prev_l + tl.sum(P, axis=-1)
        # compute O
        prev_O = current_O
        online_err = tl.exp(prev_m - m)
        current_O =  online_err[:,None] * (prev_O) + tl.dot(P, v_tile)
        # advance K pointer
        K_block_ptr = tl.advance(K_block_ptr,(K_TILE_SIZE,0))
        V_block_ptr = tl.advance(V_block_ptr,(K_TILE_SIZE,0))
        mask_block_ptr = tl.advance(mask_block_ptr,(0,K_TILE_SIZE))

    # final normalization of O
    current_O = current_O / l[:,None]
    # log(sum(exp)))
    current_l = m + tl.log(l)

    tl.store(O_block_ptr,current_O)
    tl.store(L_block_ptr,current_l)


In [19]:
import numpy as np

class flashAttentionFwd(torch.autograd.Function):
  @staticmethod
  def forward(ctx,
              Q: torch.Tensor, #(B,Nq,d)
              K: torch.Tensor, #(B,Nk,d)
              V: torch.Tensor, #(B,Nk,d)
              is_causal: bool,
              ):

    Bq = 16
    Bk = 16

    Tq = int(np.ceil(Q.shape[-2]/Bq))
    # init output O,L
    O = torch.zeros_like(Q,device = Q.device)
    L = torch.zeros(size = (Q.shape[0],Q.shape[-2]),dtype = torch.float32,device = Q.device)
    # init triangular matrix
    mask = torch.tril(torch.ones((Q.shape[-2],K.shape[-2])))
    mask[mask==0] = -1*float('Inf')
    mask[mask==1] = 0
    mask = mask.to(Q.device)

    # start a grid of (Tq,batch size)
    flash_attention_fwd[(Tq,Q.shape[0])](Q_ptr = Q,
                                        K_ptr = K,
                                        V_ptr = V,
                                        O_ptr = O,
                                        L_ptr = L,
                                        mask_ptr = mask,
                                        stride_qb = Q.shape[-2]*Q.shape[-1],
                                        stride_qq = Q.shape[-1],
                                        stride_qd = 1,
                                        stride_kb = K.shape[-2]*K.shape[-1],
                                        stride_kk = K.shape[-1],
                                        stride_kd = 1,
                                        stride_vb = V.shape[-2]*V.shape[-1],
                                        stride_vk = V.shape[-1],
                                        stride_vd = 1,
                                        stride_ob = O.shape[-2]*O.shape[-1],
                                        stride_oq = O.shape[-1],
                                        stride_od = 1,
                                        stride_lb = L.shape[-1],
                                        stride_lq = 1,
                                        N_QUERIES = Q.shape[1],
                                        N_KEYS = K.shape[1],
                                        scale = 1/np.sqrt(Q.shape[-1]),
                                        D = Q.shape[-1],
                                        Q_TILE_SIZE = Bq,
                                        K_TILE_SIZE =  Bk,
                                        is_causal = is_causal,
                                        )
    ctx.save_for_backward(L)
    return O




In [24]:
# FA without causal mask
test_flash_forward_pass_triton(False)
# FA with causal mask
test_flash_forward_pass_triton(True)

# Backward propagation using triton

